In [ ]:
import os

# ============================================================
# ENVIRONMENT SETUP — comment out the block you are NOT using
# ============================================================

# ── OPTION A: Google Colab ───────────────────────────────────
# Mounts Drive and changes to your project folder so all
# outputs (plots, pkl) save there automatically.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/ML-Assignment/attemp2'
os.chdir(DRIVE_PATH)
print(f"✓ Working directory: {DRIVE_PATH}")

# ── OPTION B: Local Jupyter ──────────────────────────────────
# Comment out Option A above and uncomment the two lines below.
# Make sure cardio_train.csv is in the same folder as this notebook.
# LOCAL_PATH = os.path.dirname(os.path.abspath("__file__"))
# os.chdir(LOCAL_PATH)

# Create output directories
os.makedirs("plots",  exist_ok=True)
os.makedirs("report", exist_ok=True)
print("✓ Output folders ready.")


In [ ]:
# ============================================================
# DATASET LOADING — comment out the block you are NOT using
# ============================================================

# ── OPTION A: Google Colab — upload file via dialog ─────────
from google.colab import files
print("Click Choose Files and select cardio_train.csv")
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"✓ Uploaded: {DATASET_PATH}")

# ── OPTION B: Local Jupyter ──────────────────────────────────
# Comment out Option A above and uncomment the line below.
# DATASET_PATH = "cardio_train.csv"  # must be in the same folder as this notebook
# print(f"✓ Dataset path: {DATASET_PATH}")


# Cardiovascular Disease Prediction
## Notebook 01: Exploratory Data Analysis & Data Preprocessing
---
**Author:** MS26912448 (Member 1)
**Dataset:** Cardiovascular Disease Dataset — Kaggle

### Objectives
This notebook covers:
1. Loading and inspecting the raw dataset
2. Exploratory Data Analysis (EDA) with visualisations
3. Data Preprocessing pipeline
4. Saving clean, scaled data for model training

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Create output directories
os.makedirs('plots', exist_ok=True)

print("✓ All libraries imported successfully.")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  sklearn : ", end='')
import sklearn; print(sklearn.__version__)

## 1. Load Dataset

The dataset is stored as a semicolon-delimited CSV. We load it using `pd.read_csv` with `sep=';'`.

In [ ]:
# Load the cardiovascular disease dataset
# NOTE: The dataset uses semicolon (;) as the delimiter
df = pd.read_csv(DATASET_PATH, sep=';')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)

## 2. Initial Data Inspection

In [ ]:
# Display column data types and non-null counts
print("=== Dataset Information ===")
df.info()

In [ ]:
# Descriptive statistics for all columns
print("=== Descriptive Statistics ===")
df.describe().round(2)

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Check data types
print("\nColumn data types:")
print(df.dtypes)

## 3. Missing Value Analysis

In [ ]:
# Check for missing values in each column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("=== Missing Values Summary ===")
print(missing_df[missing_df['Missing Count'] > 0] if missing.any()
      else "No missing values found in the dataset.")

## 4. Target Variable Analysis

The target variable `cardio` is binary:
- **0**: No cardiovascular disease
- **1**: Has cardiovascular disease

We first check for class imbalance, which affects model selection and evaluation.

In [ ]:
# Visualise target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['cardio'].value_counts().sort_index()
axes[0].bar(['No Disease (0)', 'Has Disease (1)'], counts.values,
            color=['#2196F3', '#F44336'], alpha=0.85, edgecolor='black', linewidth=0.8)
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=['No Disease', 'Has Disease'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, explode=(0.05, 0))
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Cardiovascular Disease – Target Variable', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Class 0 (No Disease): {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)")
print(f"Class 1 (Has Disease): {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)")
print(f"\nDataset is approximately balanced — no SMOTE required.")

## 5. Feature Distributions

### 5.1 Age Distribution
Age is stored in **days**. We convert to years for readability.

In [ ]:
# Age distribution (converting from days to years for visualisation)
age_years = df['age'] / 365

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(age_years, bins=35, color='#42A5F5', edgecolor='black', alpha=0.8, linewidth=0.6)
axes[0].set_title('Age Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(age_years.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {age_years.mean():.1f} yrs')
axes[0].legend()

axes[1].boxplot(age_years, vert=True, patch_artist=True,
               boxprops=dict(facecolor='#42A5F5', alpha=0.8))
axes[1].set_title('Age Boxplot', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Age (years)')

plt.suptitle('Age Feature Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Age range : {age_years.min():.0f} – {age_years.max():.0f} years")
print(f"Mean age  : {age_years.mean():.1f} years")
print(f"Std dev   : {age_years.std():.1f} years")

### 5.2 Numerical Features (Height, Weight, Blood Pressure)

In [ ]:
# Distribution of numerical features with boxplots
num_cols = ['height', 'weight', 'ap_hi', 'ap_lo']
colors   = ['#66BB6A', '#FFA726', '#EF5350', '#AB47BC']
labels   = ['Height (cm)', 'Weight (kg)', 'Systolic BP', 'Diastolic BP']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, (col, clr, lbl) in enumerate(zip(num_cols, colors, labels)):
    # Histogram
    axes[0, i].hist(df[col], bins=40, color=clr, edgecolor='black', alpha=0.8, linewidth=0.5)
    axes[0, i].set_title(lbl, fontsize=11, fontweight='bold')
    axes[0, i].set_ylabel('Frequency')
    # Boxplot
    axes[1, i].boxplot(df[col], vert=True, patch_artist=True,
                       boxprops=dict(facecolor=clr, alpha=0.8))
    axes[1, i].set_title(f'{lbl} Boxplot', fontsize=11)

plt.suptitle('Numerical Feature Distributions', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/03_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Note significant outliers in blood pressure
print("⚠ Note: Blood pressure columns contain unrealistic outliers.")
print(f"  ap_hi max: {df['ap_hi'].max()} | ap_lo min: {df['ap_lo'].min()}")
print("  These will be removed in the preprocessing step.")

### 5.3 Categorical Features

In [ ]:
# Categorical feature distributions
cat_cols = ['cholesterol', 'gluc', 'smoke', 'alco', 'active', 'gender']
label_maps = {
    'cholesterol': {1:'Normal', 2:'Above\nNormal', 3:'Well Above\nNormal'},
    'gluc':        {1:'Normal', 2:'Above\nNormal', 3:'Well Above\nNormal'},
    'smoke':       {0:'Non-Smoker', 1:'Smoker'},
    'alco':        {0:'No Alcohol', 1:'Alcohol'},
    'active':      {0:'Not Active', 1:'Active'},
    'gender':      {1:'Female', 2:'Male'}
}
clrs = ['#5C6BC0', '#26A69A', '#EC407A', '#FF7043', '#29B6F6', '#66BB6A']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (col, clr) in enumerate(zip(cat_cols, clrs)):
    counts = df[col].value_counts().sort_index()
    xlabels = [label_maps[col].get(k, str(k)) for k in counts.index]
    axes[i].bar(range(len(counts)), counts.values, color=clr, alpha=0.85,
                edgecolor='black', linewidth=0.6)
    axes[i].set_xticks(range(len(counts)))
    axes[i].set_xticklabels(xlabels, fontsize=9)
    axes[i].set_title(col.capitalize(), fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Count')
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 100, f'{v/len(df)*100:.1f}%', ha='center', fontsize=8)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/04_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Correlation Heatmap

In [ ]:
# Correlation heatmap to identify feature relationships
corr_df = df.copy()
corr_df['age_years'] = (corr_df['age'] / 365).round().astype(int)
corr_df = corr_df.drop(['id', 'age'], axis=1)

corr_matrix = corr_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(13, 10))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('plots/05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top correlations with target
target_corr = corr_matrix['cardio'].drop('cardio').sort_values(ascending=False)
print("=== Top Feature Correlations with Target (cardio) ===")
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 20)
    print(f"  {feat:<15}: {val:+.3f}  {bar}")

### 5.5 Feature vs Target Analysis

Comparing feature distributions across the two classes to understand discriminative features.

In [ ]:
# Box plots comparing features by target class
key_features = ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol']
temp_df = corr_df.copy()
temp_df['age_years'] = (df['age'] / 365).round()
temp_df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
feat_labels = ['Age (years)', 'BMI', 'Systolic BP', 'Diastolic BP', 'Cholesterol']
plot_feats  = ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol']

box_colors = ['#2196F3', '#F44336']

for i, (feat, lbl) in enumerate(zip(plot_feats, feat_labels)):
    if feat in temp_df.columns:
        groups = [temp_df[temp_df['cardio']==0][feat].dropna(),
                  temp_df[temp_df['cardio']==1][feat].dropna()]
        bp = axes[i].boxplot(groups, labels=['No Disease', 'Has Disease'],
                             patch_artist=True)
        for patch, color in zip(bp['boxes'], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        axes[i].set_title(lbl, fontsize=11, fontweight='bold')
        axes[i].set_ylabel('Value')

plt.suptitle('Feature Distribution by Cardiovascular Disease Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/06_feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Data Preprocessing

### 6.1 Age Conversion: Days → Years

The `age` column stores age in **days** (unusual format). We convert to years for interpretability
and drop the original `id` and raw `age` columns.